In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import gc
from sklearn.metrics import roc_auc_score

PROCESSED = Path("../data/processed")
REPORTS = Path("reports")
sns.set_style("whitegrid")
pd.set_option("display.max_columns", 100)

In [2]:
inst = pd.read_parquet(PROCESSED / "installments_payments.parquet")
train_target = pd.read_parquet(
    PROCESSED / "application_train_reduced.parquet",
    columns=["SK_ID_CURR", "TARGET"]
)

print(f"installments_payments: {inst.shape}")
print(f"Memoria: {inst.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
print(f"\nDtypes:\n{inst.dtypes}")
print(f"\nHead:")
print(inst.head())

installments_payments: (13605401, 8)
Memoria: 493.1 MB

Dtypes:
SK_ID_PREV                  int32
SK_ID_CURR                  int32
NUM_INSTALMENT_VERSION    float32
NUM_INSTALMENT_NUMBER       int16
DAYS_INSTALMENT           float32
DAYS_ENTRY_PAYMENT        float32
AMT_INSTALMENT            float64
AMT_PAYMENT               float64
dtype: object

Head:
   SK_ID_PREV  SK_ID_CURR  NUM_INSTALMENT_VERSION  NUM_INSTALMENT_NUMBER  \
0     1054186      161674                     1.0                      6   
1     1330831      151639                     0.0                     34   
2     2085231      193053                     2.0                      1   
3     2452527      199697                     1.0                      3   
4     2714724      167756                     1.0                      2   

   DAYS_INSTALMENT  DAYS_ENTRY_PAYMENT  AMT_INSTALMENT  AMT_PAYMENT  
0          -1180.0             -1187.0        6948.360     6948.360  
1          -2156.0             -2156.0        

In [3]:
n_curr = inst["SK_ID_CURR"].nunique()
n_prev = inst["SK_ID_PREV"].nunique()
train_with_inst = train_target["SK_ID_CURR"].isin(inst["SK_ID_CURR"]).sum()

print(f"SK_ID_CURR únicos: {n_curr:,}")
print(f"SK_ID_PREV únicos: {n_prev:,}")
print(f"Clientes train con installments: {train_with_inst:,} "
      f"({100*train_with_inst/len(train_target):.2f}%)")

rows_per_client = inst.groupby("SK_ID_CURR").size()
print(f"\n=== Pagos por cliente ===")
print(rows_per_client.describe().round(1))

SK_ID_CURR únicos: 339,587
SK_ID_PREV únicos: 997,752
Clientes train con installments: 291,643 (94.84%)

=== Pagos por cliente ===
count    339587.0
mean         40.1
std          41.1
min           1.0
25%          12.0
50%          25.0
75%          51.0
max         372.0
dtype: float64


In [4]:
print("=== Estadísticas ===")
print(inst.describe().round(1))

print("\n=== % nulos ===")
print((inst.isna().mean() * 100).round(3).sort_values(ascending=False))

=== Estadísticas ===
       SK_ID_PREV  SK_ID_CURR  NUM_INSTALMENT_VERSION  NUM_INSTALMENT_NUMBER  \
count  13605401.0  13605401.0              13605401.0             13605401.0   
mean    1903365.0    278444.9                     0.9                   18.9   
std      536202.9    102718.3                     1.0                   26.7   
min     1000001.0    100001.0                     0.0                    1.0   
25%     1434191.0    189639.0                     0.0                    4.0   
50%     1896520.0    278685.0                     1.0                    8.0   
75%     2369094.0    367530.0                     1.0                   19.0   
max     2843499.0    456255.0                   178.0                  277.0   

       DAYS_INSTALMENT  DAYS_ENTRY_PAYMENT  AMT_INSTALMENT  AMT_PAYMENT  
count       13605401.0          13602496.0      13605401.0   13602496.0  
mean           -1042.3             -1051.1         17050.9      17238.2  
std              796.2              

In [5]:
# Cuota sin pago registrado. Como todas las cuotas son pasadas, ausencia = impago real.
inst["INST_IS_UNPAID"] = (inst["DAYS_ENTRY_PAYMENT"].isna() | inst["AMT_PAYMENT"].isna()).astype("int8")

# Días de atraso: positivo = pagó tarde
inst["INST_DAYS_LATE"] = (inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]).astype("float32")

# Déficit de pago: positivo = pagó de menos
inst["INST_PAYMENT_DIFF"] = (inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]).astype("float32")

# Ratio de pago: <1 = pagó de menos
inst["INST_PAYMENT_RATIO"] = (
    inst["AMT_PAYMENT"] / inst["AMT_INSTALMENT"].replace(0, np.nan)
).astype("float32")

# Asignación del peor escenario continuo para las cuotas que no registran ningún pago
unpaid = inst["INST_IS_UNPAID"] == 1
inst.loc[unpaid, "INST_PAYMENT_DIFF"] = inst.loc[unpaid, "AMT_INSTALMENT"]   # Debe el total de la cuota
inst.loc[unpaid, "INST_PAYMENT_RATIO"] = 0.0                                 # No pagó nada de porcentaje
# Nota: inst["INST_DAYS_LATE"] queda NaN por coherencia, ya que el evento se captura con el flag.

# Indicadores lógicos compuestos: el impago cuenta directamente como tardío y como incompleto
inst["INST_IS_LATE"] = ((inst["INST_DAYS_LATE"] > 0) | unpaid).astype("int8")
inst["INST_IS_UNDERPAID"] = ((inst["INST_PAYMENT_DIFF"] > 0) | unpaid).astype("int8")

print("=== INST_DAYS_LATE (positivo = tarde) ===")
print(inst["INST_DAYS_LATE"].describe().round(1))
print(f"Pagos tardíos (incluye impagos): {inst['INST_IS_LATE'].sum():,} ({inst['INST_IS_LATE'].mean()*100:.2f}%)")

print("\n=== INST_PAYMENT_DIFF (positivo = pagó de menos) ===")
print(inst["INST_PAYMENT_DIFF"].describe().round(1))
print(f"Pagos incompletos (incluye impagos): {inst['INST_IS_UNDERPAID'].sum():,} "
      f"({inst['INST_IS_UNDERPAID'].mean()*100:.2f}%)")

C:\Users\Usuario\AppData\Local\Temp\ipykernel_22656\3610538024.py:17: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[49741.02  22849.515 48092.355 ... 43737.435    67.5   11504.25 ]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  inst.loc[unpaid, "INST_PAYMENT_DIFF"] = inst.loc[unpaid, "AMT_INSTALMENT"]   # Debe el total de la cuota


=== INST_DAYS_LATE (positivo = tarde) ===
count    13602496.0
mean           -8.8
std            25.0
min         -3189.0
25%           -14.0
50%            -6.0
75%             0.0
max          2884.0
Name: INST_DAYS_LATE, dtype: float64
Pagos tardíos (incluye impagos): 1,149,574 (8.45%)

=== INST_PAYMENT_DIFF (positivo = pagó de menos) ===
count    13605401.0
mean         -183.6
std         19124.7
min      -2630909.0
25%             0.0
50%             0.0
75%             0.0
max       2424726.5
Name: INST_PAYMENT_DIFF, dtype: float64
Pagos incompletos (incluye impagos): 1,298,398 (9.54%)


In [6]:
agg_dict = {
    "DAYS_INSTALMENT": ["count"],
    "NUM_INSTALMENT_VERSION": ["nunique"],
    "INST_IS_UNPAID": ["max", "mean", "sum"],
    "INST_DAYS_LATE": ["max", "mean", "sum"],
    "INST_PAYMENT_DIFF": ["max", "mean", "sum"],
    "INST_PAYMENT_RATIO": ["min", "mean"],
    "INST_IS_LATE": ["max", "mean", "sum"],
    "INST_IS_UNDERPAID": ["max", "mean", "sum"],
    "AMT_INSTALMENT": ["mean", "sum"],
    "AMT_PAYMENT": ["mean", "sum"],
}

inst_agg = inst.groupby("SK_ID_CURR").agg(agg_dict)
inst_agg.columns = ["INST_" + "_".join(c).upper() for c in inst_agg.columns]
inst_agg = inst_agg.rename(columns={"INST_DAYS_INSTALMENT_COUNT": "INST_N_PAYMENTS"})
inst_agg = inst_agg.reset_index()

print(f"Shape: {inst_agg.shape}")
print(inst_agg.head(3))

Shape: (339587, 24)
   SK_ID_CURR  INST_N_PAYMENTS  INST_NUM_INSTALMENT_VERSION_NUNIQUE  \
0      100001                7                                    2   
1      100002               19                                    2   
2      100003               25                                    2   

   INST_INST_IS_UNPAID_MAX  INST_INST_IS_UNPAID_MEAN  INST_INST_IS_UNPAID_SUM  \
0                        0                       0.0                        0   
1                        0                       0.0                        0   
2                        0                       0.0                        0   

   INST_INST_DAYS_LATE_MAX  INST_INST_DAYS_LATE_MEAN  INST_INST_DAYS_LATE_SUM  \
0                     11.0                 -7.285714                    -51.0   
1                    -12.0                -20.421053                   -388.0   
2                     -1.0                 -7.160000                   -179.0   

   INST_INST_PAYMENT_DIFF_MAX  INST_INST_PAYM

In [7]:
inst_target = inst_agg.merge(train_target, on="SK_ID_CURR", how="inner")
print(f"Clientes con installments + target: {inst_target.shape}")

inst_features = [c for c in inst_target.columns if c.startswith("INST_")]
auc_list = []

for col in inst_features:
    m = inst_target[col].notna()
    if m.sum() == 0 or inst_target.loc[m, col].nunique() < 2:
        continue
        
    auc_raw = roc_auc_score(inst_target.loc[m, "TARGET"], inst_target.loc[m, col])
    auc_list.append({
        "feature": col,
        "auc": round(max(auc_raw, 1 - auc_raw), 4),
        "direction": "↓ menos default" if auc_raw < 0.5 else "↑ más default",
        "pct_null": round(100 * inst_target[col].isna().mean(), 2),
    })

auc_inst = pd.DataFrame(auc_list).sort_values("auc", ascending=False)
print(f"\n=== Top features de installments por AUC ===")
print(auc_inst.to_string(index=False))
auc_inst.to_csv(REPORTS / "auc_rank_installments.csv", index=False)

Clientes con installments + target: (291643, 25)

=== Top features de installments por AUC ===
                            feature    auc       direction  pct_null
             INST_INST_IS_LATE_MEAN 0.5645   ↑ más default       0.0
       INST_INST_PAYMENT_RATIO_MEAN 0.5577 ↓ menos default       0.0
        INST_INST_PAYMENT_DIFF_MEAN 0.5554   ↑ más default       0.0
        INST_INST_IS_UNDERPAID_MEAN 0.5547   ↑ más default       0.0
            INST_INST_DAYS_LATE_SUM 0.5535   ↑ más default       0.0
            INST_INST_DAYS_LATE_MAX 0.5500   ↑ más default       0.0
         INST_INST_PAYMENT_DIFF_SUM 0.5490   ↑ más default       0.0
              INST_INST_IS_LATE_SUM 0.5484   ↑ más default       0.0
               INST_AMT_PAYMENT_SUM 0.5447 ↓ menos default       0.0
         INST_INST_IS_UNDERPAID_SUM 0.5439   ↑ más default       0.0
              INST_AMT_PAYMENT_MEAN 0.5435 ↓ menos default       0.0
              INST_INST_IS_LATE_MAX 0.5434   ↑ más default       0.0
        

In [8]:
inst_agg.to_parquet(PROCESSED / "installments_aggregated.parquet", index=False)
mem = inst_agg.memory_usage(deep=True).sum() / 1024**2
print(f"Guardado: installments_aggregated.parquet")
print(f"Shape: {inst_agg.shape} | Memoria: {mem:.1f} MB")

del inst, inst_target
gc.collect()

Guardado: installments_aggregated.parquet
Shape: (339587, 24) | Memoria: 45.3 MB


10